# EarningsLens — Stage 1 & 2: Data Loading, Preprocessing & Segmentation (2023–2025)

This notebook is a scoped version of the full Stage 1–2 pipeline, restricted to transcripts from **2023, 2024, and 2025 only**. This scope covers 531 S&P 500 companies across approximately 4,759 transcripts — consistent with the 2023–2025 window used throughout Stages 4–7.

Running on the scoped corpus reduces sentencisation time from ~15 minutes to ~3 minutes and produces a `sentences_cache.parquet` file of approximately 250k–350k sentences rather than 14.6M, which is far more practical for iterative development.

### Storage architecture

The sentence-level DataFrame is saved to `sentences_cache_2023_2025.parquet` locally — used as the input to the scoped Stage 3. The `companies` and `transcripts` tables in Supabase are updated to reflect the 2023–2025 scope.

### Pipeline flow

```
HuggingFace dataset (filtered 2023–2025)
    -> Stage 1-2  : sentences_cache_2023_2025.parquet (local)
    -> Stage 3    : fls_checkpoint.parquet (local)
    -> Stage 4-7  : fls_2023_2025.parquet (local)
    -> Stage 10   : Streamlit dashboard reads from Supabase + local parquet
```

## 0. Dependencies

In [ ]:
# %pip install -q datasets pandas nltk huggingface-hub psycopg2-binary python-dotenv yfinance requests

## 0.1 Imports & Logging

In [ ]:
# Standard library
import re
import ast
import logging
import os
from collections import Counter
from pathlib import Path
import time
import requests

# Third-party
import numpy as np
import pandas as pd
import nltk
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv
from datasets import load_dataset

c:\Users\sidsu\Desktop\Sem 4\NLP\NLPvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# NLTK sentence tokeniser models

nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)

True

In [ ]:
# Logging

logging.basicConfig(
    format="%(asctime)s [%(levelname)s] %(message)s",
    level=logging.INFO,
)
log = logging.getLogger("earningslens")

print("✓ Imports OK")

✓ Imports OK


## 1. Configuration

`YEAR_FILTER` restricts the corpus to 2023–2025 immediately after loading. This is the only change from the full pipeline — all column mappings, keyword lists, and processing logic are identical.

In [ ]:
# Supabase connection string
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    raise EnvironmentError(
        "DATABASE_URL not found. "
        "Create a .env file with: DATABASE_URL=postgresql://..."
    )
print("✓ DATABASE_URL loaded")

✓ DATABASE_URL loaded


In [ ]:
# Dataset

DATASET_NAME = "Bose345/sp500_earnings_transcripts"

In [ ]:
# Column mappings — confirmed from schema inspection
# Speaker labels live inside structured_content dicts
COL_SPEAKER  = "speaker"       # key inside each structured_content dict
COL_TEXT     = "text"          # key inside each structured_content dict
COL_TICKER   = "symbol"        # top-level ticker column (always populated)
COL_QUARTER  = "quarter"       # integer 1–4, no parsing needed
COL_YEAR     = "year"          # integer, no parsing needed
COL_COMPANY  = "company_name"  # symbol used as primary identifier
COL_COMP_ID  = "company_id"    # same rows as company_name

In [ ]:
# # Q&A boundary detection - When an operator/moderator says
# # one of these phrases, every subsequent turn in that transcript
# # is labelled 'qa'. Everything prior is 'prepared_remarks'.

# QA_TRIGGER_PHRASES = [
#     # Original phrases
#     "we will now begin the question",
#     "we'll now begin the question",
#     "we'll now open the floor",
#     "open the call for questions",
#     "open for questions",
#     "we'll now take questions",
#     "we will now open",
#     "at this time we will open",
#     "i'll now turn the call over to the operator",
#     "you may now ask questions",
#     # Extended — actual phrases found in corpus
#     "your next question comes from the line of",
#     "our next question comes from the line of",
#     "our next question will come from the line of",
#     "next question comes from the line of",
#     "your next question is from",
#     "our next question is from",
#     "we go next now to",
#     "we'll go next now to",
#     "moving on now to",
#     "and our next question comes from",
#     "the next question comes from",
#     "the next question will come from",
#     "the next question is from",
#     "our first question comes from",
#     "we'll move next to",
#     "one moment for our next question",
#     "our next question is coming from",
#     "we'll take our next question from",
#     "and we'll take our next question from",
#     "i will take our next question from",
#     "we'll take our next question from",
#     "our next question today comes from",
#     "our final question comes from",
#     "is online with your next question",
#     "and our next question is going to come from",
#     "our next question will come from",
#     "and we will take our next question from",
#     "we will take our next question from",
#     "yes, our next question will come from",
#     "and our final question will come from",
#     "and we'll go over to the line of",
#     "and we'll go to the line of",
#     "we'll go to the next line",
#     "the next question goes to",
#     "next question is coming from",
#     "and your next question today will come from",
#     "your next question today will come from",
#     "our next question comes from",
#     "you may go ahead",
#     "our next caller is",
#     "we'll take our next question from",
#     "the next question will be from",
#     "we'll go next to",
#     "next question is from",
#     "our next questions come from the line of",
#     "next we'll go to",
#     "and next we go to",
#     "and we can go to",
#     "our first question today comes from",
#     "our first question comes from",
#     "first question today comes from",
#     "and that will come from the line of",
#     "and next will be",
#     "next we go to the line of",
#     "next, we'll go to the line of",
#     "next, we will go to the line of",
#     "next we'll go to the line of",
#     "your line is open",
#     "and our next question coming from the line of",
#     "we will now take our next question from",
#     "we can now take our next question from",
#     "and we can now take our next question from",
#     "your next question is",
#     "if you would like to ask a question",
#     "we will now be conducting a question",
#     "this is from the line of",
#     "this is from",
#     "we'll take our next question. this is",
#     "and next, we go to a question from",
#     "we'll move to our next question from",
#     "we'll move to our next question from",
#     "and next we'll hear from",
#     "and next, we'll hear from",
#     "it is from",
#     "very good. that will come from",
#     "and it comes from the line of",
#     "it comes from the line of",
#     "next caller, please go ahead",
#     "we'll take our next question from",
#     "next, we'll go the line of",
#     "and next we have got",
#     "we will go now to",
#     "we have a question from",
#     "your final question is from",
#     "and we will go back to",
#     "we now have",
#     "we'll hear next from",
#     "we'll get our next question on the line from",
#     "we'll proceed with our next question on the line",
#     "next, we'll go to",
#     "next question will be from",
#     "will take our last question from",
#     "is on the line with a question",
#     "we will move next to",
#     "we will move to",
#     "we will go to",
# ]

In [ ]:
QA_TRIGGER_PHRASES = [
    # ── Explicit session openers ──────────────────────────────────────────
    "we will now begin the question",
    "we'll now begin the question",
    "we'll now open the floor",
    "open the call for questions",
    "open for questions",
    "we'll now take questions",
    "we will now open",
    "at this time we will open",
    "i'll now turn the call over to the operator",
    "you may now ask questions",
    "we will now be conducting a question",
    "if you would like to ask a question",      # classic operator instructions preamble

    # ── "comes from" family ───────────────────────────────────────────────
    "your next question comes from the line of",
    "our next question comes from the line of",
    "our next question will come from the line of",
    "next question comes from the line of",
    "your next question is from",
    "our next question is from",
    "and our next question comes from",
    "the next question comes from",
    "the next question will come from",
    "the next question is from",
    "our first question comes from",
    "our first question today comes from",
    "first question today comes from",
    "our next question today comes from",
    "our final question comes from",
    "and our final question will come from",
    "our next question is coming from",
    "and our next question coming from the line of",
    "and our next question is going to come from",
    "our next questions come from the line of",
    "and our next question is from",

    # ── "will come from" / "coming from" family ───────────────────────────
    "our next question will come from",
    "yes, our next question will come from",
    "the next question will be from",
    "next question will be from",
    "next question is from",
    "next question is coming from",
    "and your next question today will come from",
    "your next question today will come from",
    "and that will come from the line of",
    "very good. that will come from",

    # ── "take our next question" family ───────────────────────────────────
    "we'll take our next question from",
    "and we'll take our next question from",
    "we will take our next question from",
    "and we will take our next question from",
    "we will now take our next question from",
    "we can now take our next question from",
    "and we can now take our next question from",
    "we'll take the next question from",
    "will take our last question from",
    "we'll get our next question on the line from",
    "we'll proceed with our next question on the line",
    "i will take our next question from",

    # ── "go to" family ────────────────────────────────────────────────────
    "we go next now to",
    "we'll go next now to",
    "we'll go next to",
    "next we'll go to",
    "next, we'll go to the line of",
    "next, we'll go to",
    "next, we'll go the line of",           # corpus typo variant (missing "to")
    "next, we will go to the line of",
    "next we'll go to the line of",
    "next we go to the line of",
    "and we'll go over to the line of",
    "and we'll go to the line of",
    "we'll go to the next line",
    "and we can go to",
    "we will go now to",
    "and next we go to",
    "and next, we go to a question from",
    "next, we go to the line of",
    "we'll go to the line of",

    # ── "move to" family ──────────────────────────────────────────────────
    "we'll move next to",
    "we'll move to our next question from",
    "we will move next to",
    "moving on now to",

    # ── "hear from" / "have" family ───────────────────────────────────────
    "and next we'll hear from",
    "and next, we'll hear from",
    "we'll hear next from",
    "we have a question from",
    "we have the next question from the line of",
    "and next we have got",
    "our next caller is",
    "next caller, please go ahead",

    # ── "goes to" / "is from" / "it comes from" ───────────────────────────
    "the next question goes to",
    "your final question is from",
    "and it comes from the line of",
    "it comes from the line of",
    "is on the line with a question",
    "is online with your next question",

    # ── "go to" / "go next" / "go back" ──────────────────────────────────
    "and we will go back to",
    "one moment for our next question",

    # ── "line is open" / "line is live" ──────────────────────────────────
    "your line is open",
    "your line is now live",
    "your line is now open",

    # ── "this is from the line of" (specific enough to keep) ─────────────
    "this is from the line of",
    "we'll take our next question. this is from",

    # ── "next, we'll go" / "next we go" ──────────────────────────────────
    "and next, we'll go to",
    "and next we'll go to",
    "and next will be",                     # keep - operator context only
    "we will move to our next question",
]

In [ ]:
# Speaker role keyword heuristics — matched against lowercased label strings
# Operator is checked before executive to avoid misclassifying edge cases
# such as 'Conference Director' which contains an executive keyword

EXEC_KEYWORDS = [
    "ceo", "cfo", "coo", "cto", "president", "chief",
    "officer", "founder", "chairman", "chairwoman",
    "vice president", "vp ", "evp", "svp", "director",
    "executives",                           # bulk label across many transcripts
    "unidentified company representative",  # unambiguously exec-side
]

ANALYST_KEYWORDS = [
    "analyst", "research", "securities", "capital", "partners",
    "bank", "morgan", "goldman", "barclays", "citi", "ubs",
    "jefferies", "cowen", "raymond", "piper", "mizuho",
    "deutsche", "credit suisse", "wolfe", "longbow", "keybanc",
    "jpmorgan", "bofa", "merrill", "rbc", "wells fargo",
    "evercore", "bernstein", "baird", "stifel", "oppenheimer",
]

OPERATOR_KEYWORDS = ["operator", "moderator", "coordinator", "conference"]

In [ ]:
# Processing - Minimum sentence length in characters: shorter strings are artifacts
# (e.g. "Thank you.", "Yes.") and are dropped before FLS classification

MIN_SENTENCE_LEN = 15

In [ ]:
# ── Year scope ────────────────────────────────────────────────────────────────
# Restricts the corpus to 2023–2025 — consistent with all downstream stages
YEAR_FILTER = [2023, 2024, 2025]

In [ ]:
# Output path
OUTPUT_PATH = Path("sentences_cache_2023_2025.parquet")

---
## Stage 1 — Load & Filter Raw Dataset

The full dataset is loaded from HuggingFace (cached locally after first run) and immediately filtered to 2023–2025 rows. This reduces the working corpus from 33,362 transcripts to approximately 4,759 before any processing begins.

In [ ]:
log.info("Loading %s...", DATASET_NAME)
ds  = load_dataset(DATASET_NAME, split="train")
raw = ds.to_pandas()
log.info("Full dataset loaded: %d rows", len(raw))

# Apply year filter immediately — keeps memory usage low throughout Stage 2
raw = raw[raw[COL_YEAR].isin(YEAR_FILTER)].copy().reset_index(drop=True)
log.info("After year filter: %d rows (%s)", len(raw), YEAR_FILTER)

print(f"Rows after year filter   : {len(raw):,}")
print(f"Unique tickers           : {raw[COL_TICKER].nunique():,}")
print(f"Year distribution:")
print(raw[COL_YEAR].value_counts().sort_index().to_string())

2026-05-04 07:07:38,643 [INFO] Loading Bose345/sp500_earnings_transcripts...
2026-05-04 07:07:39,289 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/Bose345/sp500_earnings_transcripts/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-05-04 07:07:39,309 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Bose345/sp500_earnings_transcripts/dc9b436a0d8e02e62e393e1616a497fcd54b8b6c/README.md "HTTP/1.1 200 OK"
2026-05-04 07:07:39,548 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/Bose345/sp500_earnings_transcripts/resolve/dc9b436a0d8e02e62e393e1616a497fcd54b8b6c/sp500_earnings_transcripts.py "HTTP/1.1 404 Not Found"
2026-05-04 07:07:40,325 [INFO] HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/Bose345/sp500_earnings_transcripts/Bose345/sp500_earnings_transcripts.py "HTTP/1.1 404 Not Found"
2026-05-04 07:07:40,719 [INFO] HTTP Request: GET https://huggingface.co/api/datasets/Bose345/sp500_earnings_t

Rows after year filter   : 4,759
Unique tickers           : 531
Year distribution:
year
2023    2079
2024    2033
2025     647


### 1.1 Schema & Null Audit

In [ ]:
print("=== Columns ===")
print(list(raw.columns))

print("\n=== dtypes ===")
print(raw.dtypes.to_string())

print("\n=== Null counts ===")
print(raw.isnull().sum().to_string())

=== Columns ===
['symbol', 'quarter', 'year', 'date', 'content', 'structured_content', 'company_name', 'company_id']

=== dtypes ===
symbol                    str
quarter                 int64
year                    int64
date                      str
content                   str
structured_content     object
company_name              str
company_id            float64

=== Null counts ===
symbol                  0
quarter                 0
year                    0
date                    0
content                 0
structured_content      0
company_name          209
company_id            209


### 1.2 Structured Content & Speaker Inspection

In [ ]:
def parse_structured(val) -> list:
    """
    Deserialise one structured_content value into a list of speaker-turn dicts.
    structured_content is stored as a numpy.ndarray by HuggingFace datasets.
    """
    if isinstance(val, list):
        return val
    if hasattr(val, "tolist"):
        return val.tolist()
    try:
        return ast.literal_eval(val)
    except Exception:
        return []

In [ ]:
log.info("Extracting speaker labels from structured_content...")
speaker_counter = Counter()
for val in raw["structured_content"]:
    for turn in parse_structured(val):
        spk = turn.get(COL_SPEAKER, "")
        if spk:
            speaker_counter[spk] += 1

print(f"Unique speaker labels : {len(speaker_counter):,}")
print(f"Total speaker turns   : {sum(speaker_counter.values()):,}")

2026-05-04 07:08:01,561 [INFO] Extracting speaker labels from structured_content...


Unique speaker labels : 8,064
Total speaker turns   : 323,997


In [ ]:
def classify_label(speaker: str) -> str:
    s = speaker.lower()
    if any(k in s for k in OPERATOR_KEYWORDS): return "operator"
    if any(k in s for k in EXEC_KEYWORDS):     return "executive"
    if any(k in s for k in ANALYST_KEYWORDS):  return "analyst"
    return "unknown"

In [ ]:
role_counts = Counter()
for spk, count in speaker_counter.items():
    role_counts[classify_label(spk)] += count

total = sum(role_counts.values())
print("Role distribution (keyword pass):")
for role, count in role_counts.items():
    print(f"  {role:<12}: {count:>8,}  ({count/total*100:.1f}%)")

Role distribution (keyword pass):
  operator    :   59,686  (18.4%)
  unknown     :  260,340  (80.4%)
  analyst     :    2,626  (0.8%)
  executive   :    1,345  (0.4%)


### 1.3 Stage 1 Summary

In [ ]:
print("=" * 52)
print("  EarningsLens \u2014 Stage 1 Complete (2023\u20132025)")
print("=" * 52)
print(f"\nTranscripts loaded    : {len(raw):,}")
print(f"Unique tickers        : {raw[COL_TICKER].nunique():,}")
print(f"Year range            : {raw[COL_YEAR].min()}\u2013{raw[COL_YEAR].max()}")
print(f"Total speaker turns   : {sum(role_counts.values()):,}")
print(f"\n\u2192 Ready for Stage 2")

  EarningsLens — Stage 1 Complete (2023–2025)

Transcripts loaded    : 4,759
Unique tickers        : 531
Year range            : 2023–2025
Total speaker turns   : 323,997

→ Ready for Stage 2


---
## Stage 2 — Preprocessing & Segmentation

Identical processing logic to the full pipeline — transcript ID synthesis, structured content explosion, two-pass speaker role classification, Q&A boundary detection, text cleaning, and sentencisation. Applied to the 2023–2025 subset only.

### 2.1 Transcript ID Synthesis

In [ ]:
raw["transcript_id"] = (
    raw[COL_TICKER].astype(str) + "_"
    + raw[COL_YEAR].astype(str)  + "_Q"
    + raw[COL_QUARTER].astype(str)
)

In [ ]:
dupes = raw["transcript_id"].duplicated().sum()
ok    = "\u2713 all unique" if dupes == 0 else "\u26a0 duplicates present"
print(f"Unique transcript IDs : {raw['transcript_id'].nunique():,}")
print(f"Duplicate IDs         : {dupes}  {ok}")

Unique transcript IDs : 4,759
Duplicate IDs         : 0  ✓ all unique


In [ ]:
print(f"Sample IDs            : {raw['transcript_id'].head(5).tolist()}")

Sample IDs            : ['A_2023_Q4', 'A_2023_Q3', 'A_2023_Q2', 'A_2023_Q1', 'A_2024_Q4']


### 2.2 Company Name Enrichment

In [ ]:
# --------------------------------------------------------------------------
# Step 1: In-dataset propagation
#   Many tickers appear in multiple rows; if ANY row has a name, copy it to
#   all null rows for that ticker. Free and instant.
# --------------------------------------------------------------------------
ticker_to_name = (
    raw[raw[COL_COMPANY].notna() & (raw[COL_COMPANY].astype(str).str.strip() != "")]
    .groupby(COL_TICKER)[COL_COMPANY]
    .first()
    .to_dict()
)

null_before = raw[COL_COMPANY].isna().sum()
raw[COL_COMPANY] = raw.apply(
    lambda r: ticker_to_name.get(r[COL_TICKER], r[COL_COMPANY])
    if pd.isna(r[COL_COMPANY]) else r[COL_COMPANY],
    axis=1,
)
null_after_propagation = raw[COL_COMPANY].isna().sum()
log.info(
    "Name propagation: %d nulls -> %d nulls (filled %d)",
    null_before, null_after_propagation, null_before - null_after_propagation,
)

# Tickers still without a name after propagation
still_missing_tickers = (
    raw[raw[COL_COMPANY].isna()][COL_TICKER].unique().tolist()
)

print(f"Companies that have tickers with null names   : {len(still_missing_tickers)}")

2026-05-04 07:08:02,632 [INFO] Name propagation: 209 nulls -> 209 nulls (filled 0)


Companies that have tickers with null names   : 35


In [ ]:
def yfinance_lookup(tickers: list) -> dict:
    """Returns {ticker: longName} for tickers yfinance can resolve."""
    import yfinance as yf
    resolved = {}
    batch_size = 50
    for i in range(0, len(tickers), batch_size):
        batch = tickers[i: i + batch_size]
        try:
            data = yf.Tickers(" ".join(batch))
            for t in batch:
                try:
                    info = data.tickers[t].info
                    name = info.get("longName") or info.get("shortName")
                    if name:
                        resolved[t] = name
                except Exception:
                    pass
            time.sleep(0.3)
        except Exception as e:
            log.warning("yfinance batch error: %s", e)
    return resolved

In [ ]:
yf_resolved = {}
if still_missing_tickers:
    log.info("yfinance lookup for %d tickers...", len(still_missing_tickers))
    yf_resolved = yfinance_lookup(still_missing_tickers)
    log.info("  yfinance resolved %d / %d", len(yf_resolved), len(still_missing_tickers))
else:
    log.info("No tickers needed yfinance lookup.")

2026-05-04 07:08:02,657 [INFO] yfinance lookup for 35 tickers...
2026-05-04 07:08:09,504 [ERROR] HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CMA"}}}
2026-05-04 07:08:18,285 [INFO]   yfinance resolved 28 / 35


In [ ]:
unresolved = set(still_missing_tickers) - set(yf_resolved.keys())
if unresolved:
    log.warning("Unresolved tickers: %s", sorted(unresolved))

2026-05-04 07:08:18,298 [WARNING] Unresolved tickers: ['CMA', 'CTLT', 'DISH', 'FRC', 'MRO', 'PXD', 'WRK']


In [ ]:
# --------------------------------------------------------------------------
# Step 3: Manually update the remaining unresolved tickers with hardcoded mappings. This is a
# --------------------------------------------------------------------------
MANUAL_TICKER_MAP = {
    "CMA": "Comerica Incorporated",
    "CTLT": "Catalent Inc",
    "DISH": "DISH Network Corporation",
    "FRC": "First Republic Bank",
    "MRO": "Marathon Oil Corporation",
    "PXD": "Pioneer Natural Resources Company",
    "WRK": "WestRock Company"
}

In [ ]:
manual_tickers = [t for t in still_missing_tickers if t not in yf_resolved]

manual_resolved = {
    t: MANUAL_TICKER_MAP[t]
    for t in manual_tickers
    if t in MANUAL_TICKER_MAP
}

log.info(
    "Manual fallback resolved %d / %d",
    len(manual_resolved),
    len(manual_tickers)
)

2026-05-04 07:08:18,320 [INFO] Manual fallback resolved 7 / 7


In [ ]:
# --------------------------------------------------------------------------
# Step 4: Apply all resolved names back to raw DataFrame
# --------------------------------------------------------------------------

all_resolved = {**yf_resolved, **manual_resolved}

def _fill_name(row):
    if pd.notna(row[COL_COMPANY]) and str(row[COL_COMPANY]).strip():
        return row[COL_COMPANY]
    return all_resolved.get(row[COL_TICKER], row[COL_COMPANY])

In [ ]:

raw[COL_COMPANY] = raw.apply(_fill_name, axis=1)

null_final = raw[COL_COMPANY].isna().sum()
truly_unresolved = [t for t in still_missing_tickers if t not in all_resolved]

print(f"\nEnrichment summary")
print(f"  Nulls before              : {len(still_missing_tickers)}")
print(f"  Filled by propagation     : {null_before - null_after_propagation:,}")
print(f"  Filled by yfinance        : {len(yf_resolved):,}")
print(f"  Filled by manual mapping  : {len(manual_resolved):,}")
print(f"  Remaining nulls           : {null_final:,}")
if truly_unresolved:
    print(f"  Unresolved tickers        : {truly_unresolved}")
print(f"\n✓ company_name enrichment complete")



Enrichment summary
  Nulls before              : 35
  Filled by propagation     : 0
  Filled by yfinance        : 28
  Filled by manual mapping  : 7
  Remaining nulls           : 0

✓ company_name enrichment complete


### 2.2 Structured Content Explosion

In [ ]:
log.info("Exploding structured_content into speaker-turn rows...")

turn_records = []
for _, row in raw.iterrows():
    for turn in parse_structured(row["structured_content"]):
        speaker = turn.get(COL_SPEAKER, "")
        text    = turn.get(COL_TEXT, "")
        if not text or not str(text).strip():
            continue
        turn_records.append({
            "transcript_id": row["transcript_id"],
            "ticker"       : row[COL_TICKER],
            "company_name" : row[COL_COMPANY],
            "quarter"      : int(row[COL_QUARTER]),
            "year"         : int(row[COL_YEAR]),
            "speaker"      : speaker,
            "text"         : str(text).strip(),
        })

2026-05-04 07:08:18,825 [INFO] Exploding structured_content into speaker-turn rows...


In [ ]:
turns_df = pd.DataFrame(turn_records)
log.info(
    "Explosion complete: %d turns from %d transcripts",
    len(turns_df), turns_df["transcript_id"].nunique()
)
print(f"Turns shape : {turns_df.shape}")
print(f"Columns     : {list(turns_df.columns)}")

2026-05-04 07:08:29,078 [INFO] Explosion complete: 323587 turns from 4759 transcripts


Turns shape : (323587, 7)
Columns     : ['transcript_id', 'ticker', 'company_name', 'quarter', 'year', 'speaker', 'text']


### 2.3 Two-Pass Speaker Role Classification

In [ ]:
# First pass — keyword heuristics
turns_df["speaker_role"] = turns_df["speaker"].apply(classify_label)

print("First-pass role distribution:")
print(turns_df["speaker_role"].value_counts().to_string())

First-pass role distribution:
speaker_role
unknown      260099
operator      59654
analyst        2540
executive      1294


### 2.4 Q&A Boundary Detection

Uses a positional index approach — avoids the `groupby().apply()` column-dropping issue encountered in the full pipeline.

In [ ]:
log.info(
    "Detecting Q&A boundaries across %d transcripts...",
    turns_df["transcript_id"].nunique()
)

turns_df = turns_df.reset_index(drop=True)

# Build trigger mask: operator turns containing a Q&A trigger phrase
is_operator  = turns_df["speaker_role"] == "operator"
trigger_mask = turns_df["text"].str.lower().apply(
    lambda t: any(phrase in t for phrase in QA_TRIGGER_PHRASES)
)
is_trigger = is_operator & trigger_mask

# Per transcript: positional index of first trigger row
first_trigger = (
    turns_df[is_trigger]
    .groupby("transcript_id")
    .apply(lambda g: g.index.min())
    .to_dict()
)

2026-05-04 07:08:30,356 [INFO] Detecting Q&A boundaries across 4759 transcripts...


In [ ]:
def get_segment(row) -> str:
    threshold = first_trigger.get(row["transcript_id"], float("inf"))
    return "qa" if row.name >= threshold else "prepared_remarks"

In [ ]:
turns_df["segment_type"] = turns_df.apply(get_segment, axis=1)

print("Segment type distribution:")
print(turns_df["segment_type"].value_counts().to_string())

Segment type distribution:
segment_type
qa                  277146
prepared_remarks     46441


In [ ]:
qa_pct = (turns_df["segment_type"] == "qa").mean() * 100
if qa_pct < 5:
    log.warning("Only %.1f%% of turns labelled qa — check trigger phrases", qa_pct)

In [ ]:
# Second pass — resolve name-only unknowns by segment position
# prepared_remarks unknowns -> executive | qa unknowns -> analyst
def resolve_unknown(row) -> str:
    if row["speaker_role"] != "unknown":
        return row["speaker_role"]
    return "executive" if row["segment_type"] == "prepared_remarks" else "analyst"

In [ ]:
turns_df["speaker_role"] = turns_df.apply(resolve_unknown, axis=1)

print("Final role distribution after two-pass classification:")
print(turns_df["speaker_role"].value_counts().to_string())

Final role distribution after two-pass classification:
speaker_role
analyst      224032
operator      59654
executive     39901


In [ ]:
qa_tx = turns_df[turns_df["segment_type"] == "qa"]["transcript_id"].nunique()
total_tx = turns_df["transcript_id"].nunique()
print(f"Transcripts with Q&A detected: {qa_tx:,} / {total_tx:,} ({qa_tx/total_tx*100:.1f}%)")

Transcripts with Q&A detected: 4,550 / 4,759 (95.6%)


In [ ]:
# # Diagnostic: sample actual operator turns from after transcript midpoint

# import random

# # Transcripts where NO qa boundary was detected yet
# no_qa_ids = [
#     tid for tid, idx in first_trigger.items()
#     if idx == float("inf")
# ]
# # Also include transcripts with no trigger at all
# all_ids = turns_df["transcript_id"].unique().tolist()
# no_qa_ids = [tid for tid in all_ids if tid not in first_trigger]

# print(f"Transcripts with no Q&A boundary detected: {len(no_qa_ids):,}")

# sample_ids = random.sample(no_qa_ids, min(50, len(no_qa_ids)))

# operator_turns = []
# for tid in sample_ids:
#     tx = turns_df[turns_df["transcript_id"] == tid].reset_index(drop=True)
#     midpoint = len(tx) // 2
#     after_mid = tx[tx.index >= midpoint]
#     op_turns = after_mid[after_mid["speaker_role"] == "operator"]["text"].tolist()
#     operator_turns.extend(op_turns[:3])

# print(f"\nSample operator turns from second half of undetected transcripts:\n")
# for i, turn in enumerate(operator_turns[:30], 1):
#     print(f"{i:>2}. {turn[:120]}")

### 2.5 Text Cleaning

In [ ]:
def clean_text(text: str) -> str:
    """Strip [annotations] and normalise whitespace."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"\s+",     " ", text)
    return text.strip()

In [ ]:
turns_df["clean_text"] = turns_df["text"].apply(clean_text)

before    = len(turns_df)
turns_df  = turns_df[turns_df["clean_text"].str.len() > 0].reset_index(drop=True)
print(f"Turns dropped (empty after cleaning) : {before - len(turns_df):,}")
print(f"Turns remaining                      : {len(turns_df):,}")

Turns dropped (empty after cleaning) : 120
Turns remaining                      : 323,467


### 2.6 Sentencisation

Each turn is split into individual sentences using NLTK Punkt. Runtime is approximately 1–3 minutes for the 2023–2025 subset.

In [ ]:
def sentencise_turn(row: pd.Series) -> list:
    """
    Split one cleaned speaker turn into sentence-level records.
    Sentences below MIN_SENTENCE_LEN characters are discarded.
    """
    records = []
    for i, sent in enumerate(nltk.sent_tokenize(row["clean_text"])):
        sent = sent.strip()
        if len(sent) < MIN_SENTENCE_LEN:
            continue
        records.append({
            "transcript_id" : row["transcript_id"],
            "ticker"        : row["ticker"],
            "company_name"  : row["company_name"],
            "quarter"       : row["quarter"],
            "year"          : row["year"],
            "speaker"       : row["speaker"],
            "speaker_role"  : row["speaker_role"],
            "segment_type"  : row["segment_type"],
            "sentence_index": i,
            "sentence"      : sent,
        })
    return records

In [ ]:
log.info("Sentencising %d turns...", len(turns_df))

all_records = []
for _, row in turns_df.iterrows():
    all_records.extend(sentencise_turn(row))

sentences_df = pd.DataFrame(all_records)

log.info(
    "Sentencisation complete: %d sentences from %d transcripts",
    len(sentences_df), sentences_df["transcript_id"].nunique()
)

2026-05-04 07:09:03,621 [INFO] Sentencising 323467 turns...


2026-05-04 07:10:29,851 [INFO] Sentencisation complete: 2082188 sentences from 4759 transcripts


### 2.7 Output Validation

In [ ]:
print("=" * 56)
print("  EarningsLens \u2014 Stage 2 Output (2023\u20132025)")
print("=" * 56)
print(f"\nTotal sentences      : {len(sentences_df):,}")
print(f"Unique transcripts   : {sentences_df['transcript_id'].nunique():,}")
print(f"Unique tickers       : {sentences_df['ticker'].nunique():,}")
print(f"Year range           : {sentences_df['year'].min()}\u2013{sentences_df['year'].max()}")

print("\n\u2500\u2500 Speaker role breakdown \u2500\u2500")
for role, count in sentences_df["speaker_role"].value_counts().items():
    print(f"  {role:<12}: {count:>8,}  ({count/len(sentences_df)*100:.1f}%)")

print("\n\u2500\u2500 Segment type breakdown \u2500\u2500")
for seg, count in sentences_df["segment_type"].value_counts().items():
    print(f"  {seg:<20}: {count:>8,}  ({count/len(sentences_df)*100:.1f}%)")

print("\n\u2500\u2500 Sample executive / prepared_remarks sentences \u2500\u2500")
sample = (
    sentences_df[
        (sentences_df["speaker_role"] == "executive") &
        (sentences_df["segment_type"] == "prepared_remarks")
    ]["sentence"].sample(5, random_state=42)
)
for i, s in enumerate(sample, 1):
    print(f"  {i}. {s[:130]}")

  EarningsLens — Stage 2 Output (2023–2025)

Total sentences      : 2,082,188
Unique transcripts   : 4,759
Unique tickers       : 531
Year range           : 2023–2025

── Speaker role breakdown ──
  analyst     : 1,213,443  (58.3%)
  executive   :  741,224  (35.6%)
  operator    :  127,521  (6.1%)

── Segment type breakdown ──
  qa                  : 1,315,589  (63.2%)
  prepared_remarks    :  766,599  (36.8%)

── Sample executive / prepared_remarks sentences ──
  1. That’s really the only material equity needs that we have planned at this stage.
  2. I'll now turn it over to Shyam to share more details.
  3. Participants on today's call will include Mark Lashier, President, CEO; Kevin Mitchell, CFO; Brian Mandell, Marketing and Commerci
  4. We posted another quarter of strong cash flow generation, which is indicative of the quality of our earnings and cash conversion.
  5. And then turning to franchisee and your question about how we support franchisees, you're right that our U.S. fr

---
## Supabase — Schema & Metadata Insert

The full pipeline schema is created (or confirmed) in Supabase and `companies` and `transcripts` are populated with the 2023–2025 scope. All `INSERT` statements use `ON CONFLICT DO NOTHING` — safe to rerun.

In [ ]:
conn   = psycopg2.connect(DATABASE_URL)
cursor = conn.cursor()

cursor.execute("SELECT current_database(), version();")
db_name, version = cursor.fetchone()
print(f"Connected to Supabase: {db_name}  |  {version.split(',')[0]}")

Connected to Supabase: postgres  |  PostgreSQL 17.6 on aarch64-unknown-linux-gnu


In [ ]:
# Insert companies
companies = (
    sentences_df[["ticker", "company_name"]]
    .drop_duplicates(subset="ticker")
    .dropna(subset=["ticker"])
)
execute_values(
    cursor,
    "INSERT INTO companies (ticker, company_name) VALUES %s ON CONFLICT (ticker) DO NOTHING",
    companies[["ticker", "company_name"]].values.tolist(),
)
conn.commit()

cursor.execute("SELECT ticker, company_id FROM companies")
ticker_to_id = {row[0]: row[1] for row in cursor.fetchall()}
print(f"Companies : {len(ticker_to_id):,}")

Companies : 531


In [ ]:
# Insert transcripts
transcripts = (
    sentences_df[["transcript_id", "ticker", "quarter", "year"]]
    .drop_duplicates(subset="transcript_id")
    .copy()
)
transcripts["company_id"] = transcripts["ticker"].map(ticker_to_id)

execute_values(
    cursor,
    "INSERT INTO transcripts (transcript_id, company_id, quarter, year) VALUES %s ON CONFLICT (transcript_id) DO NOTHING",
    transcripts[["transcript_id", "company_id", "quarter", "year"]].values.tolist(),
)
conn.commit()
print(f"Transcripts : {len(transcripts):,}")

Transcripts : 4,759


In [ ]:
# Verify row counts
for table in ["companies", "transcripts", "commitments", "sentiment_scores"]:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"  {table:<20}: {cursor.fetchone()[0]:>8,} rows")

cursor.close()
conn.close()
print("\n✓ Supabase connection closed")

  companies           :      531 rows
  transcripts         :    4,759 rows
  commitments         :        0 rows
  sentiment_scores    :        0 rows

✓ Supabase connection closed


## Save sentences_df to Parquet

`sentences_cache_2023_2025.parquet` is the input to the scoped Stage 3. It is also the handoff file if the kernel is restarted between sessions.

In [ ]:
sentences_df.to_parquet(OUTPUT_PATH, index=False)

print(f"\u2713 {OUTPUT_PATH} saved")
print(f"  Rows    : {len(sentences_df):,}")
print(f"  Columns : {list(sentences_df.columns)}")
print(f"\n Ready for Stage 3 (2023\u20132025)")

✓ sentences_cache_2023_2025.parquet saved
  Rows    : 2,082,188
  Columns : ['transcript_id', 'ticker', 'company_name', 'quarter', 'year', 'speaker', 'speaker_role', 'segment_type', 'sentence_index', 'sentence']

 Ready for Stage 3 (2023–2025)
